# Datasets y DataLoaders


El código para procesar muestras de datos puede volverse desordenado y difícil de mantener. Idealmente queremos que nuestro código de dataset esté **desacoplado** de nuestro código de entrenamiento del modelo para mejor legibilidad y modularidad. 

PyTorch proporciona dos primitivas de datos fundamentales:
- **`torch.utils.data.DataLoader`**: Envuelve un iterable alrededor del Dataset para facilitar el acceso a las muestras
- **`torch.utils.data.Dataset`**: Almacena las muestras y sus etiquetas correspondientes

Estas primitivas te permiten usar datasets precargados así como tus propios datos.

## Datasets Precargados

Las librerías de dominio de PyTorch proporcionan varios datasets precargados (como FashionMNIST) que heredan de `torch.utils.data.Dataset` e implementan funciones específicas para datos particulares. Pueden usarse para prototipar y hacer benchmarks de tu modelo.

Puedes encontrarlos aquí:
- **Image Datasets** (Datasets de imágenes)
- **Text Datasets** (Datasets de texto)
- **Audio Datasets** (Datasets de audio)

## Cargar un Dataset

Aquí hay un ejemplo de cómo cargar el dataset Fashion-MNIST desde TorchVision. Fashion-MNIST es un dataset de imágenes de artículos de Zalando que consiste en:
- **60,000 ejemplos de entrenamiento**
- **10,000 ejemplos de prueba**
- Cada ejemplo comprende una imagen en escala de grises de **28×28** píxeles
- Una etiqueta asociada de una de **10 clases**

### Parámetros para cargar FashionMNIST:

- **`root`**: ruta donde se almacenan los datos de entrenamiento/prueba
- **`train`**: especifica si es dataset de entrenamiento o prueba
- **`download=True`**: descarga los datos de internet si no están disponibles en root
- **`transform`** y **`target_transform`**: especifican las transformaciones de características y etiquetas

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cargar datos de entrenamiento
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

# Cargar datos de prueba
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

print(f"Tamaño del conjunto de entrenamiento: {len(training_data)}")
print(f"Tamaño del conjunto de prueba: {len(test_data)}")
print(f"Forma de una imagen: {training_data[0][0].shape}")
print(f"Etiqueta de la primera imagen: {training_data[0][1]}")

## Iterar y Visualizar el Dataset

Podemos indexar Datasets manualmente como una lista: `training_data[index]`. Usamos matplotlib para visualizar algunas muestras en nuestros datos de entrenamiento.

In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

In [ ]:
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

## Crear un Dataset Personalizado para tus archivos

Una clase Dataset personalizada debe implementar tres funciones: **`__init__`**, **`__len__`**, y **`__getitem__`**. 

Observa esta implementación; las imágenes de FashionMNIST se almacenan en un directorio `img_dir`, y sus etiquetas se almacenan por separado en un archivo CSV `annotations_file`.

En las siguientes secciones, desglosaremos lo que sucede en cada una de estas funciones.

In [ ]:
import os
import pandas as pd
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

### `__init__`

La función `__init__` se ejecuta una vez al instanciar el objeto Dataset. Inicializamos:
- El directorio que contiene las imágenes
- El archivo de anotaciones
- Ambas transformaciones (cubiertas con más detalle en la siguiente sección)

El archivo `labels.csv` se ve así:
```
tshirt1.jpg, 0
tshirt2.jpg, 0
......
ankleboot999.jpg, 9
```

In [ ]:
def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
    self.img_labels = pd.read_csv(annotations_file)
    self.img_dir = img_dir
    self.transform = transform
    self.target_transform = target_transform

### `__len__`

La función `__len__` devuelve el número de muestras en nuestro dataset.

**Ejemplo:**

In [ ]:
def __len__(self):
    return len(self.img_labels)

### `__getitem__`

La función `__getitem__` carga y devuelve una muestra del dataset en el índice `idx` dado. 

Basándose en el índice:
1. Identifica la ubicación de la imagen en el disco
2. Convierte eso a un tensor usando `decode_image`
3. Recupera la etiqueta correspondiente de los datos csv en `self.img_labels`
4. Llama a las funciones de transformación en ellos (si aplica)
5. Devuelve la imagen tensor y la etiqueta correspondiente en una tupla

In [ ]:
def __getitem__(self, idx):
    img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
    image = decode_image(img_path)
    label = self.img_labels.iloc[idx, 1]
    if self.transform:
        image = self.transform(image)
    if self.target_transform:
        label = self.target_transform(label)
    return image, label

## Preparar tus datos para entrenamiento con DataLoaders

El Dataset recupera las características y etiquetas de nuestro dataset **una muestra a la vez**. Mientras entrenamos un modelo, típicamente queremos:
- Pasar muestras en **"minibatches"** (mini-lotes)
- **Reorganizar** los datos en cada época para reducir el sobreajuste del modelo
- Usar **`multiprocessing`** de Python para acelerar la recuperación de datos

`DataLoader` es un iterable que abstrae esta complejidad para nosotros en una API fácil.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

## Iterar a través del DataLoader

Hemos cargado ese dataset en el DataLoader y podemos iterar a través del dataset según sea necesario. Cada iteración a continuación devuelve un lote de `train_features` y `train_labels` (conteniendo `batch_size=64` características y etiquetas respectivamente). 

Porque especificamos `shuffle=True`, después de iterar sobre todos los lotes, los datos se mezclan (para un control más fino sobre el orden de carga de datos, echa un vistazo a **Samplers**).

In [ ]:
# Mostrar imagen y etiqueta
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

## Ventajas de usar DataLoader

### 1. **Procesamiento por Lotes (Batching)**
En lugar de procesar una imagen a la vez, DataLoader agrupa múltiples muestras en lotes. Esto es más eficiente porque:
- Las GPUs están optimizadas para operaciones matriciales grandes
- Reduce el número de actualizaciones de parámetros
- Proporciona estimaciones más estables del gradiente

### 2. **Mezclado (Shuffling)**
Mezclar los datos ayuda a:
- Prevenir que el modelo aprenda patrones basados en el orden de los datos
- Mejorar la generalización
- Reducir el sobreajuste

### 3. **Carga Paralela**
Con `num_workers > 0`, DataLoader puede cargar datos en paralelo mientras el modelo está entrenando, lo que acelera significativamente el proceso de entrenamiento.

### 4. **Gestión Automática de Memoria**
DataLoader maneja eficientemente la memoria al cargar solo un lote a la vez, no todo el dataset.